# Feature importance
CV-aware SHAP analysis for XGBoost + SelectKBest under Stratified 5-Fold CV. Execute cells top to bottom. Only cells marked **[CONFIGURE]** require changes.

## 1. Environment Setup **[OPTIONAL]**
Mounts Google Drive and installs dependencies when running on Colab. Skip if running locally.

> **NOTE: requires moving `PROJECT` folder to Google Drive.**

In [ ]:
# COLAB
# Change runtime type to GPU
import sys
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd '/content/drive/My Drive/PROJECT'

    # Install deps
    !pip install -q numpy pandas torch scikit-learn xgboost shap nilearn seaborn matplotlib

    # Add project modules
    sys.path.insert(0, './product/src')

## 2. Imports

In [ ]:
import sys
import re
import os
import numpy as np
import pandas as pd
import shap
import xgboost as xgb
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import matplotlib.lines as mlines
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import StratifiedKFold
from nilearn import image, plotting

from data_io.save_load_dataset import load_dataset, mixed_subset
from utils.paths import get_project_root

## 3. Configuration **[CONFIGURE]**

Set `PREFIX` to match the dataset used in `run_experiments.ipynb`.

| Variable | Options | Description |
|---|---|---|
| `PREFIX` | `'1100'`, `'female'`, `'male'`, `'mixed'` | Dataset to analyse |
| `K` | integer | Number of features selected by SelectKBest |
| `N_SPLITS` | integer | Number of outer CV folds |
| `TOP_N` | integer | Number of top features to display in plots |
| `OUTPUT_DIR` | path | Directory to save outputs |

In [ ]:
PREFIX     = "1100"       # dataset prefix: "1100", "female", "male", "mixed"
K          = 4000         # SelectKBest k
N_SPLITS   = 5            # outer CV folds
SEED       = 42
TOP_N      = 10           # features to display in beeswarm and connectome
OUTPUT_DIR = "./shap_output"

os.makedirs(OUTPUT_DIR, exist_ok=True)

## 4. Load Dataset

Loads the processed dataset. Switch to `mixed_subset()` if using the sex-balanced subset.

> **NOTE: using `mixed` requires downloading and processing the female and male age-matched subsets first.**

In [ ]:
root = get_project_root()
X, X_raw, y, metadata, feature_labels = load_dataset(PREFIX, verbose=True)
# X, y, metadata, feature_labels = mixed_subset()  # uncomment for mixed subset

column_names = [
    re.sub(r'[\[\]<>\s:]+', '_', str(name)).strip('_')
    for name in feature_labels['feature_name']
]
n_features_total = X.shape[1]

## 5. CV-aware SHAP analysis

For each outer fold:
1. SelectKBest is fit on training data only.
2. XGBoost is fit on the selected training features.
3. SHAP `TreeExplainer` is applied to the held-out test fold.
4. SHAP values are mapped back to the original 19,900-feature space.

SHAP values are accumulated across folds for averaging. Only held-out test fold data is used throughout.

In [ ]:
shap_abs_sum   = np.zeros(n_features_total)
shap_sign_sum  = np.zeros(n_features_total)
fold_counts    = np.zeros(n_features_total)

# Accumulators for beeswarm: one sparse array per participant across all folds
beeswarm_shap_rows = []
beeswarm_conn_rows = []

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X, y)):
    print(f"\nFold {fold_idx + 1}/{N_SPLITS}")

    X_train, X_test = X[train_idx], X[test_idx]
    y_train         = y[train_idx]

    # Feature selection
    selector      = SelectKBest(score_func=f_classif, k=K)
    X_train_sel   = selector.fit_transform(X_train, y_train)
    X_test_sel    = selector.transform(X_test)
    selected_mask = selector.get_support()
    selected_idx  = np.where(selected_mask)[0]

    # Fit XGBoost
    model = xgb.XGBClassifier(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        random_state=SEED,
        eval_metric="logloss",
        use_label_encoder=False,
    )
    model.fit(X_train_sel, y_train, verbose=False)

    # SHAP on held-out test fold only
    explainer   = shap.TreeExplainer(model)
    shap_values = explainer(X_test_sel)

    sv = shap_values.values
    if sv.ndim == 3:
        sv = sv[:, :, 1]    # class-1 SHAP values, shape (n_test, K)

    # Accumulate CV averages (mapped back to original feature space)
    fold_abs  = np.abs(sv).mean(axis=0)
    fold_sign = sv.mean(axis=0)

    shap_abs_sum[selected_idx]  += fold_abs
    shap_sign_sum[selected_idx] += fold_sign
    fold_counts[selected_idx]   += 1

    # Map SHAP and connectivity values back to the original feature space
    sv_full   = np.zeros((len(test_idx), n_features_total))
    conn_full = np.zeros((len(test_idx), n_features_total))
    sv_full[:, selected_idx]   = sv
    conn_full[:, selected_idx] = X_test_sel

    beeswarm_shap_rows.append(sv_full)
    beeswarm_conn_rows.append(conn_full)

    top5 = np.argsort(fold_abs)[-5:][::-1]
    print(f"  Top 5 features this fold:")
    for rank, fi in enumerate(top5, 1):
        orig_idx = selected_idx[fi]
        print(f"    {rank}. {column_names[orig_idx]}  |SHAP|={fold_abs[fi]:.5f}")

## 6. Aggregate results

Averages SHAP values across folds and prints the top features.

In [ ]:
with np.errstate(invalid='ignore'):
    mean_abs_shap  = np.where(fold_counts > 0, shap_abs_sum  / fold_counts, 0.0)
    mean_sign_shap = np.where(fold_counts > 0, shap_sign_sum / fold_counts, 0.0)

# Each participant appears exactly once (in their held-out fold)
beeswarm_shap_all = np.vstack(beeswarm_shap_rows)   # (n_total, n_features_total)
beeswarm_conn_all = np.vstack(beeswarm_conn_rows)   # (n_total, n_features_total)

top_indices = np.argsort(mean_abs_shap)[-TOP_N:][::-1]

results = []
for rank, orig_idx in enumerate(top_indices, 1):
    row       = feature_labels.iloc[orig_idx]
    direction = 'ASD ↑' if mean_sign_shap[orig_idx] > 0 else 'ASD ↓'
    results.append({
        'Rank':        rank,
        'Region pair': row.get('label_feature_name', column_names[orig_idx]),
        'Network':     row.get('network', ''),
        'Direction':   direction,
        'Mean |SHAP|': round(float(mean_abs_shap[orig_idx]), 6),
    })

df_results = pd.DataFrame(results)
print(f"\n=== Top {TOP_N} features by CV-averaged mean |SHAP| ({PREFIX}) ===")
print(df_results.to_string(index=False))

## 7. Save feature importance CSV

Saves the full ranked feature importance table to `OUTPUT_DIR`.

In [ ]:
csv_path = f"{OUTPUT_DIR}/shap_mean_abs_{PREFIX}.csv"
full_df  = pd.DataFrame({
    'feature_name':     column_names,
    'mean_abs_shap':    mean_abs_shap,
    'mean_signed_shap': mean_sign_shap,
    'folds_selected':   fold_counts,
})
full_df = full_df.sort_values('mean_abs_shap', ascending=False)
full_df.to_csv(csv_path, index=False)
print(f"Full feature importance saved to: {csv_path}")

## 8. Beeswarm plot

Generates a SHAP beeswarm plot from accumulated held-out test fold data. Each dot represents one participant in their held-out fold. Colour indicates connectivity strength, normalised to the range shown on the colour bar.

In [ ]:
print("Generating beeswarm plot from held-out test fold data...")

# Top features ranked descending for display
top_orig_indices_desc = np.argsort(mean_abs_shap)[-TOP_N:][::-1]

# Subset accumulated arrays to top features only
shap_subset = beeswarm_shap_all[:, top_orig_indices_desc]   # (n_total, TOP_N)
conn_subset = beeswarm_conn_all[:, top_orig_indices_desc]   # (n_total, TOP_N)

display_names = []
for orig_idx in top_orig_indices_desc:
    row = feature_labels.iloc[orig_idx]
    display_names.append(row.get('abbrev_feature_name', column_names[orig_idx]))

# Colour map
slate_blue  = "#4573C4"
salmon_red  = "#E35959"
custom_cmap = mcolors.LinearSegmentedColormap.from_list(
    "slate_salmon", [slate_blue, "#FFFFFF", salmon_red]
)

# Reverse order so highest-ranked feature is at the top
shap_exp = shap.Explanation(
    values        = shap_subset[:, ::-1],
    data          = conn_subset[:, ::-1],
    feature_names = display_names[::-1],
)

fig = plt.figure(figsize=(12, 6), dpi=300)
shap.plots.beeswarm(
    shap_exp,
    max_display=TOP_N,
    color=custom_cmap,
    plot_size=None,
    show=False,
)

ax = plt.gca()
ax.set_xlabel("SHAP value (impact on model output)", fontsize=10, labelpad=10)
ax.tick_params(axis='y', pad=-10, labelsize=8)
ax.tick_params(axis='x', labelsize=8)
ax.grid(True, linestyle='--', alpha=0.4, zorder=0)

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(0.8)
    spine.set_color('#333333')

if len(fig.axes) > 1:
    cb_ax = fig.axes[1]
    cb_ax.tick_params(labelsize=7)
    cb_ax.set_ylabel("Connectivity strength", fontsize=9, labelpad=10)
    cb_ax.set_yticks([0, 0.5, 1])
    cb_ax.set_yticklabels(['-1', '0', '1'], fontsize=8)

plt.subplots_adjust(left=0.35, right=0.95, top=0.9, bottom=0.15)
plt.title(f'CV-averaged SHAP | {PREFIX}', fontsize=10)

beeswarm_path = f"{OUTPUT_DIR}/shap_beeswarm_{PREFIX}.pdf"
fig.savefig(beeswarm_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Beeswarm saved to: {beeswarm_path}")

## 9. LaTeX table

Generates a LaTeX `\\table` environment for the top features. Participants are split by SHAP direction into TD (SHAP < 0) and ASD (SHAP > 0) groups. Mean FC is normalised to [-1, 1] relative to the full distribution of each feature, matching the beeswarm colour scale.

In [ ]:
def build_shap_table_latex(mean_abs_shap, beeswarm_shap_all,
                           beeswarm_conn_all,
                           feature_labels, dataset_name, top_n=5):
    top_indices = np.argsort(mean_abs_shap)[-top_n:][::-1]

    rows = []
    for idx in top_indices:
        row      = feature_labels.iloc[idx]
        sv_col   = beeswarm_shap_all[:, idx]
        fc_col   = beeswarm_conn_all[:, idx]

        # Participants where the feature was selected in their fold
        selected_mask = fc_col != 0
        neg_mask = (sv_col < 0) & selected_mask   # TD side
        pos_mask = (sv_col > 0) & selected_mask   # ASD side

        mean_shap_neg = sv_col[neg_mask].mean() if neg_mask.any() else 0.0
        mean_shap_pos = sv_col[pos_mask].mean() if pos_mask.any() else 0.0

        def mean_fc_norm(mask):
            if not mask.any():
                return 0.0
            fc_full = fc_col[selected_mask]
            fc_min, fc_max = fc_full.min(), fc_full.max()
            if fc_max == fc_min:
                return 0.0
            return (2 * (fc_col[mask] - fc_min) / (fc_max - fc_min) - 1).mean()

        rows.append({
            'Region Pair': row['label_feature_name'],
            'Network':     row['network'],
            'shap_neg':    f"{mean_shap_neg:.4f}",
            'fc_neg':      f"{mean_fc_norm(neg_mask):.4f}",
            'shap_pos':    f"{mean_shap_pos:.4f}",
            'fc_pos':      f"{mean_fc_norm(pos_mask):.4f}",
        })

    lines = [
        r'\begin{table}[ht]',
        r'\centering',
        r'\small',
        (
            r'\caption{Top ' + str(top_n) +
            r' functional connectivity features by CV-averaged mean absolute SHAP value, ' +
            dataset_name + r' dataset, ordered by mean $|\text{SHAP}|$ descending. '
            r'Participants are split by SHAP direction. '
            r'Under $\xleftarrow{\text{TD}}$: mean SHAP and mean FC for participants '
            r'for whom the feature decreased the ASD prediction (SHAP $< 0$). '
            r'Under $\xrightarrow{\text{ASD}}$: mean SHAP and mean FC for participants '
            r'for whom the feature increased the ASD prediction (SHAP $> 0$). '
            r'Mean FC is normalised to $[-1, 1]$ relative to the full distribution '
            r'of that feature, corresponding to the beeswarm colour scale. '
            r'All values are computed from held-out test fold data only.}'
        ),
        r'\label{tab:shap_' + dataset_name.lower() + r'}',
        r'\resizebox{\textwidth}{!}{',
        r'\begin{tabular}{llcccc}',
        r'\toprule',
        (r' & & \multicolumn{2}{c}{$\xleftarrow{\text{TD}}$} & '
         r'\multicolumn{2}{c}{$\xrightarrow{\text{ASD}}$} \\'),
        r'\cmidrule(lr){3-4} \cmidrule(lr){5-6}',
        (r'Region pair & Network & '
         r'Mean SHAP & Mean FC & '
         r'Mean SHAP & Mean FC \\'),
        r'\midrule',
    ]

    for row in rows:
        region  = row['Region Pair'].replace('_', r'\_').replace('<->', r'$\leftrightarrow$')
        network = row['Network'].replace('<->', r'$\leftrightarrow$')
        lines.append(
            f"{region} & {network} & "
            f"{row['shap_neg']} & {row['fc_neg']} & "
            f"{row['shap_pos']} & {row['fc_pos']} \\\\"
        )

    lines += [r'\bottomrule', r'\end{tabular}', r'}', r'\end{table}']
    return '\n'.join(lines)


print(build_shap_table_latex(mean_abs_shap, beeswarm_shap_all,
                              beeswarm_conn_all,
                              feature_labels, PREFIX))

## 10. Connectome plot

Plots a connectome showing the ROIs involved in the top features. Node colour indicates brain network (from the AAL atlas). Update the three file paths below to match your local or Drive paths.

> **NOTE: run `scripts/run_scripts.ipynb` to generate `coords.csv`**

In [ ]:
root = get_project_root()
data = root / 'product' / 'data'
atlas_path = data / 'external' / 'cc200_roi_atlas.nii.gz'
aal_map    = pd.read_csv(data / 'external' / 'AAL_map.csv')
coords_df  = pd.read_csv(data / 'processed' / 'coords.csv')

print("Generating connectome plot...")

atlas_img = image.load_img(atlas_path)
coords    = plotting.find_parcellation_cut_coords(atlas_path)
print(f"Extracted {len(coords)} region coordinates.")

label_to_network = dict(zip(aal_map['NOTATION'], aal_map['NETWORK']))
unique_networks  = aal_map['NETWORK'].unique()
colours          = sns.color_palette("Set3", len(unique_networks))
network_palette  = {net: colours[i] for i, net in enumerate(unique_networks)}

node_colors = []
for label in coords_df['label']:
    network = label_to_network.get(label)
    node_colors.append(network_palette.get(network, (0.5, 0.5, 0.5)))

# Colour map for edges
slate_blue        = "#4573C4"
salmon_red        = "#E35959"
STATIC_EDGE_COLOUR = "#4A4A4A"
DISPLAY_MODE      = "z"

# Identify ROIs involved in top features
involved_rois = np.unique(
    feature_labels.iloc[top_orig_indices_desc][['roi_i', 'roi_j']].values.astype(int)
)
roi_mapping      = {old: new for new, old in enumerate(involved_rois)}
n_filtered       = len(involved_rois)
filtered_coords  = coords[involved_rois]
filtered_node_colors = [node_colors[i] for i in involved_rois]

adj_static = np.zeros((n_filtered, n_filtered))
adj_neg    = np.zeros((n_filtered, n_filtered))

for orig_idx in top_orig_indices_desc:
    orig_i = int(feature_labels.iloc[orig_idx]['roi_i'])
    orig_j = int(feature_labels.iloc[orig_idx]['roi_j'])
    i, j   = roi_mapping[orig_i], roi_mapping[orig_j]
    adj_static[i, j] = adj_static[j, i] = 1

fig_conn = plt.figure(figsize=(12, 6), dpi=300)

display = plotting.plot_connectome(
    adjacency_matrix=adj_static,
    node_coords=filtered_coords,
    node_color=filtered_node_colors,
    node_size=220,
    edge_cmap=mcolors.ListedColormap([STATIC_EDGE_COLOUR]),
    edge_threshold=0.1,
    display_mode=DISPLAY_MODE,
    black_bg=False,
    colorbar=False,
    figure=fig_conn,
    title=None,
    alpha=1.0,
    node_kwargs={"edgecolor": "black", "linewidth": 1, "alpha": 1.0},
    edge_kwargs={"linewidth": 4, "alpha": 0.8, "color": STATIC_EDGE_COLOUR},
)

display.add_graph(
    adjacency_matrix=adj_neg,
    node_coords=filtered_coords,
    node_color=filtered_node_colors,
    node_size=220,
    edge_cmap=mcolors.LinearSegmentedColormap.from_list(
        "neg_edges", ["#AED6F1", "olivedrab"]
    ),
    edge_vmin=0, edge_vmax=1, edge_threshold=0,
    edge_kwargs={"linewidth": 6},
    node_kwargs={"edgecolor": "black", "linewidth": 1, "alpha": 0.9},
)

network_handles = [
    mpatches.Patch(facecolor=color, label=net, edgecolor="black", linewidth=0.8)
    for net, color in network_palette.items()
]

main_ax = display.axes["z"].ax
main_ax.legend(
    handles=network_handles,
    title="Network / Connectivity",
    title_fontsize='large', fontsize='large',
    loc="upper center", bbox_to_anchor=(0.5, -0.08),
    ncol=4, frameon=False, columnspacing=1.0, labelspacing=0.6,
)

connectome_path = f"{OUTPUT_DIR}/shap_connectome_{PREFIX}.pdf"
plt.subplots_adjust(bottom=0.22)
fig_conn.savefig(connectome_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Connectome saved to: {connectome_path}")